Before the Transformer architecture, **Recurrent Neural Networks (RNNs)** were widely used for Natural Language Processing (NLP) tasks such as machine translation, text generation, and language modeling. RNNs are **not Large Language Models (LLMs)**; they are a neural network architecture that was commonly used to build language models before Transformers.

In machine translation, words cannot be translated independently because the meaning of a word depends on the context of the entire sentence. Translating one word at a time would often produce grammatically incorrect or unnatural translations.

To capture context, RNNs use a **hidden state**, which acts as the model's memory. As the encoder reads each input word sequentially, it updates its hidden state using the current word and the previous hidden state. In an encoder-decoder (Seq2Seq) model, the encoder produces hidden state representations for the input sequence, and the decoder uses this contextual information to generate the translated sentence one word at a time. The mathematical details of how the hidden state is updated are not important for this course.

One major limitation of RNNs is that they process tokens sequentially, making training slow. They also struggle to remember information from words that appeared much earlier in long sentences because the hidden state has limited capacity. As a result, important context can be lost.

To address this problem, researchers introduced the **attention mechanism**. Instead of relying only on the encoder's final hidden state, the decoder could attend to **all encoder hidden states** and focus on the most relevant input words when generating each output word. This significantly improved translation quality, especially for long sentences.

Later, the paper **"Attention Is All You Need"** showed that recurrence was no longer necessary. It introduced the **Transformer** architecture, which replaces RNNs with **self-attention**. In self-attention, each token computes how much attention it should pay to every other token in the sequence, allowing the model to capture both nearby and long-range relationships efficiently. Since all tokens can be processed in parallel, Transformers train much faster and scale much better than RNNs, making them the foundation of modern Large Language Models (LLMs) such as GPT.


Attention machnisms is used to understand how each token attend with other token in context(something like how each token allign with other). for exxample "Your Journey starts with one step" Journey start are more allign then Journey one. So we already convert token to embedding in last chapter, think how we can decide one token is allign to another mathematically. Answer is dot product.

The dot product between two vectors $a$ and $b$ is

$$
a \cdot b = |a||b|\cos(\theta)
$$

where

- $|a|$ is the magnitude of vector $a$.
- $|b|$ is the magnitude of vector $b$.
- $\theta$ is the angle between the two vectors.

The dot product measures how well two vectors are aligned.

- **Positive:** $\cos(\theta) > 0$
- **Zero:** $\cos(\theta) = 0$
- **Negative:** $\cos(\theta) < 0$

Maximum positive value:

$$
\theta = 0^\circ,\quad \cos(\theta)=1
$$

Maximum negative value:

$$
\theta = 180^\circ,\quad \cos(\theta)=-1
$$

In Transformers, the attention score is computed as

$$
Q \cdot K
$$


Now that we have computed the **attention scores**, the next step is to **normalize** them. The normalized values are called **attention weights**. These weights determine how much attention each token should give to every other token.

Normalization has two important benefits:

1. It converts the raw attention scores into values that are easier to interpret.
2. It keeps the values in a stable range, which helps the model train more effectively.

A simple way to normalize the scores is to divide each value by the sum of all values:

$$
w_i = \frac{v_i}{\sum_j v_j}
$$

For example, if the attention scores are `[2, 3, 5]`, the normalized weights would be:

$$
\left[\frac{2}{10}, \frac{3}{10}, \frac{5}{10}\right]
=====================================================

[0.2,;0.3,;0.5]
$$

Although this approach works for positive values, it has several problems:

* If some scores are **negative**, the normalized values may also become negative, which does not make sense for attention weights.
* If the sum of the scores is **zero**, the normalization is undefined.
* Simple normalization preserves the relative differences between scores, whereas Softmax amplifies larger scores and suppresses smaller ones. This makes the model focus more sharply on the most relevant tokens.

Instead, Transformers use the **Softmax** function:

$$
\text{Softmax}(v_i) =
\frac{e^{v_i}}
{\sum_j e^{v_j}}
$$

Softmax has several advantages:

* All output values are **positive**.
* The attention weights always **sum to 1**, so they can be interpreted as probabilities.
* Larger attention scores receive proportionally larger weights, allowing the model to focus more on important tokens.
* Softmax is smooth and differentiable, making it well-suited for gradient-based optimization during training.

In practice, we do **not** compute the Softmax formula manually. Instead, we use **PyTorch's** `torch.softmax()` function:

```python
attention_weights = torch.softmax(attention_scores, dim=-1)
```

`torch.softmax()` is numerically stable. Internally, it applies optimizations (such as subtracting the maximum score before exponentiation) to prevent overflow and underflow, ensuring accurate results even when the attention scores are very large or very small.


Now that we have the **attention weights**, the next step is to compute the **context vector**. The context vector is the final output of the attention mechanism and is passed to the subsequent layers of the Transformer.

To compute the context vector, each token's **Value (V) vector** (or embedding in our simplified explanation) is multiplied by its corresponding attention weight, and then all the weighted vectors are summed together.

Mathematically,

$$
\text{Context Vector}
=====================

\sum_i \alpha_i V_i
$$

where:

* $\alpha_i$ is the attention weight for token $i$.
* $V_i$ is the Value vector of token $i$.

For example, suppose the attention weights for the current token are:

```text
[0.1, 0.7, 0.2]
```

The context vector is computed as:

$$
0.1V_1 + 0.7V_2 + 0.2V_3
$$

Notice that the second token contributes much more than the other two because it has the highest attention weight.

### Intuition

A context vector can be thought of as an **updated representation of the current token**.

Initially, a token embedding contains only the meaning of that token. After the attention mechanism, the token "looks at" the other tokens in the sentence and gathers the information that is most relevant to it.

As a result, the context vector no longer represents only the current token. Instead, it represents the current token **in the context of the entire sentence**.

For example, consider the sentence:

> **"The animal didn't cross the street because it was tired."**

The embedding for **"it"** alone is ambiguous. During attention, the token **"it"** gives a higher weight to **"animal"** than to words like **"street"** or **"because"**. The resulting context vector therefore contains information primarily from **"animal"**, helping the model understand that **"it"** refers to the animal.

In other words, the context vector is a weighted combination of information from all tokens, where more relevant tokens contribute more and less relevant tokens contribute less. This allows each token to have a context-aware representation rather than an isolated meaning.


Lets implement Self attention without weights

In [8]:
import torch

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

def calculate_context_embeddings(inputs):
    attn_scores = inputs @ inputs.T
    attn_weights = torch.softmax(attn_scores, dim=-1)
    print("weights", attn_weights)
    context_embd =  attn_weights @ inputs
    print("context", context_embd)


calculate_context_embeddings(inputs)


weights tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
context tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


# Self-Attention with Trainable Weights

## Self-Attention without Trainable Weights

Without trainable weights, self-attention uses a fixed formula to compute the context vector:

$$
\text{Attention}(X)=\text{softmax}(XX^T)X
$$

where **X** is the input embedding matrix.

Since there are no learnable parameters, the attention pattern is completely determined by the input embeddings. The model can produce contextual embeddings, but it **cannot learn or improve** during training because there are no parameters to update.

### Example

Consider the sentence:

> **The cat crossed the road because it was hungry.**

Suppose the initial embedding of **"it"** is more similar to **"road"** than **"cat"**.

The attention scores might look like:

| Token | Attention Score |
|------|---------------:|
| Cat | 0.35 |
| Road | 0.65 |

The context vector for **"it"** is therefore influenced more by **"road"**.

If the correct reference is actually **"cat"**, the prediction will be incorrect. During backpropagation, the model cannot fix this behavior because there are **no trainable parameters** inside the attention mechanism. The same input embeddings will always produce the same attention scores.

---

## Self-Attention with Trainable Weights

Instead of directly using the input embeddings, the model first transforms them into **Query (Q)**, **Key (K)**, and **Value (V)** vectors.

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

where:

- $W_Q$ = Query weight matrix
- $W_K$ = Key weight matrix
- $W_V$ = Value weight matrix

These matrices are trainable and are updated during backpropagation.

The attention computation becomes:

$$
\text{Attention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

---

## Why Trainable Weights Are Important

The matrices $W_Q$, $W_K$, and $W_V$ act as **learnable transformations**.

Suppose the model predicts the next word incorrectly because **"it"** attends more to **"road"** than **"cat"**.

Initially:

| Token | Attention Score |
|------|---------------:|
| Cat | 0.35 |
| Road | 0.65 |

During training, gradient descent updates $W_Q$, $W_K$, and $W_V$.

After training, the attention scores may become:

| Token | Attention Score |
|------|---------------:|
| Cat | 0.82 |
| Road | 0.12 |

Now the context vector for **"it"** is influenced more by **"cat"**, allowing the model to make the correct prediction.

The model has **learned** how to compute better attention rather than relying on the fixed similarities of the original embeddings.



# Why Query, Key, and Value?

The **attention algorithm does not change**.

Simple attention:

$$
\text{Attention}(X)=\text{softmax}(XX^T)X
$$

Trainable self-attention:

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
$$

$$
\text{Attention}(Q,K,V)=
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

The only difference is that **the input embedding is first transformed into three different representations.**
## Simple Attention

Suppose the embedding for **cat** is

| Embedding Features |
|---|
| animal |
| furry |
| pet |
| noun |
| living thing |

Simple attention uses **exactly this same vector** for:

- finding similar tokens
- deciding attention
- passing information to the next layer

So,

```
Embedding
   ↓
Search
Match
Send
```

Everything uses the same representation.
## Trainable Self-Attention

Instead of using the same embedding everywhere, the model learns three projections.

### Query (What am I looking for?)

```
cat embedding
    ↓ WQ
[noun, subject]
```

### Key (How should others find me?)

```
cat embedding
    ↓ WK
[noun, subject]
```

### Value (What information should I send?)

```
cat embedding
    ↓ WV
[animal, furry, pet, living]
```

Notice that the **Value is not a new fact**.

The original embedding already contains information about the word.

The weight matrix **learns which parts of that embedding should be emphasized and passed to the next layer.**
## Why is Value Different?

Imagine another token attends to **cat**.

Should it receive only

```
noun
subject
```

Probably not.

It may be more useful to receive

```
animal
furry
pet
living
```

The model learns this automatically.

So:

- **Query** learns what to search for.
- **Key** learns how to be matched.
- **Value** learns what information should be transmitted.
## Visual Comparison

### Simple Attention

```
Embedding
   │
   ├── Search
   ├── Match
   └── Send information
```

Everything is identical.

---

### Trainable Self-Attention

```
Embedding
      │
 ┌────┼────┐
 │    │    │
WQ   WK   WV
 │    │    │
 Q    K    V

Q -> Search
K -> Match
V -> Information sent to the next layer
```

The algorithm is still attention.

The improvement comes from **learning three specialized views of the same embedding** instead of using one fixed representation everywhere.


# Scaled Dot-Product Attention

The attention score is computed as:

$$
\text{Score} = QK^T
$$

The final attention formula is:

$$
\text{Attention}(Q,K,V)
=
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

---

## Why do we divide by $\sqrt{d_k}$?

As the embedding dimension ($d_k$) increases, the dot product between the Query and Key vectors also increases because it is the **sum of many products**.

For example:

- 2 dimensions → sum of 2 products
- 64 dimensions → sum of 64 products
- 512 dimensions → sum of 512 products

Larger dimensions naturally produce larger attention scores.

---

## Why is that a problem?

The attention scores are passed through the **Softmax** function.

Large scores make Softmax extremely confident.

Example:

```
Scores: [1000, 995, 990]

Softmax ≈ [1.0, 0.0, 0.0]
```

When Softmax becomes too sharp, its gradients become very small (saturation), making training slower and less stable.

---

## Why exactly $\sqrt{d_k}$?

Assume each element of the Query and Key vectors has:

- Mean = 0
- Variance = 1

Then the variance of their dot product is

$$
\text{Var}(QK^T)=d_k
$$

and the standard deviation is

$$
\sqrt{d_k}
$$

Since the **typical magnitude** of the dot product grows as $\sqrt{d_k}$, dividing by $\sqrt{d_k}$ keeps the attention scores at a consistent scale regardless of the embedding dimension.

---

## Intuition

Without scaling:

```
Higher Dimension
        ↓
 Larger Dot Product
        ↓
 Softmax Saturates
        ↓
 Small Gradients
        ↓
 Poor Training
```

With scaling:

```
Higher Dimension
        ↓
 Larger Dot Product
        ↓
 Divide by √dₖ
        ↓
 Stable Scores
        ↓
 Healthy Gradients
        ↓
 Better Training
```

---

## Key Takeaway

The scaling factor

$$
\frac{1}{\sqrt{d_k}}
$$

prevents the dot products from becoming too large as the embedding dimension increases. This keeps the Softmax output well-behaved, preserves useful gradients during backpropagation, and makes training stable.

In [9]:
import torch.nn as nn
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, qkv_bias)

    def forward(self, x):
        """ let input has 5 token and context len 3 then input dim(5, 3).
        d_in should be 3 and let say d_out is 2 typically both d_in and d_out same in real world model
        dim(5, 3) @ dim(3, 2) => dim(5, 2)
        Q.K transpose = dim(5, 2) @ dim(2, 5) => dim(5, 5)
        (Q.K transpose).V = dim(5, 5) @ dim(5, 2) => dim(5, 2) context emb dim """

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context = attn_weights @ values
        return context
        

In [10]:
torch.manual_seed(789)
sa_v2 = SelfAttention(inputs.shape[-1], 2)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


Casual Attention 

we modify the standard self-attention mechanism to create a causal attention mechanism, which is essential for developing an LLM in the subsequent chapters. Causal attention, also known as masked attention, is a specialized form of self-attention.

It restricts a model to only consider previous and current inputs in a sequence when processing any given token. This is in contrast to the standard self-attention mechanism, which allows access to the entire input sequence at once. Consequently, when computing attention scores, the causal attention mechanism ensures that the model only factors in tokens that occur at or before the current token in
the sequence.

we can take advantage of a mathematical property of the softmax function and implement the computation of the masked attention weights more efficiently in fewer steps. The softmax function converts its inputs into a probability distribution. When negative
infinity values (-∞) are present in a row, the softmax function treats them as zero probability. (Mathematically, this is because e
-∞ approaches 0.)We can implement this more efficient masking "trick" by creating a mask with 1's above the diagonal and then replacing these 1's with negative infinity (-inf) values:


Dropout:

Dropout in deep learning is a technique where randomly selected hidden layer units are ignored during training, effectively "dropping" them out. This method helps prevent overfitting by ensuring that a model does not become overly reliant on any specific set of hidden layer units. It's important to emphasize that dropout is only used during training and is disabled afterward.

In the transformer architecture, including models like GPT, dropout in the attention mechanism is typically applied in two specific areas: after calculating the attention scores or after applying the attention weights to the value vectors. Here, we will apply the dropout mask after computing the attention weights.

When applying dropout to an attention weight matrix with a rate of 50%, half of the elements in the matrix are randomly set to zero. To compensate for the reduction in active elements, the values of the remaining elements in the matrix are scaled up by a factor of
1/0.5 =2. This scaling is crucial to maintain the overall balance of the attention weights, ensuring that the average influence of the attention mechanism remains consistent during both the training and inference phases.

In [13]:
import torch.nn as nn

"""
let say batch is 6
each has context size = 5
each token has emb size = 2
input size = (6,5,2)

mostly d_in and d_out will be same, but here we say d_in = 2 and d_out = 3

then with batch size of 6

input @ W_keys = (6, 5, 2) @ (2, 3) =  (6, 5, 3)
same for values and keys

attn scores = queries @ keys.T = (6, 5, 3) @ (6, 3, 5) = (6, 5, 5)
context = attn_weights @ values = (6, 5, 5) @ (6, 5, 3) = (6, 5, 3)
"""
class CasualAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, kqv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_keys = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_values = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_queries = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length),
            diagonal=1)
            )
        
    def forward(self, input):
        print(self.mask)
        print(f"Input               : {input.shape}")
        print(self.W_keys.weight.shape)
        b, num_tokens, d_in = input.shape 
        queries = self.W_queries(input)
        keys = self.W_keys(input)
        values = self.W_values(input)

        print(f"Queries             : {queries.shape}")
        print(f"Keys                : {keys.shape}")
        print(f"Values              : {values.shape}")

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_( 
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        print(f"Attention Weights   : {attn_weights.shape}")
        context_vec = attn_weights @ values
        print(f"Context Vector      : {context_vec.shape}")
        return context_vec




In [14]:
torch.manual_seed(123)

batch_size = 6
context_length = 5
d_in = 2
d_out = 3

x = torch.randn(batch_size, context_length, d_in)

model = CasualAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=0.0
)

output = model(x)

print("\nFinal Output Shape:", output.shape)
print(output)

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])
Input               : torch.Size([6, 5, 2])
torch.Size([3, 2])
Queries             : torch.Size([6, 5, 3])
Keys                : torch.Size([6, 5, 3])
Values              : torch.Size([6, 5, 3])
Attention Weights   : torch.Size([6, 5, 5])
Context Vector      : torch.Size([6, 5, 3])

Final Output Shape: torch.Size([6, 5, 3])
tensor([[[ 0.0588, -0.0837, -0.2497],
         [ 0.0273,  0.0999, -0.2318],
         [ 0.0208, -0.0409, -0.0786],
         [ 0.0134,  0.0169, -0.0871],
         [ 0.0565,  0.0432, -0.3430]],

        [[ 0.1915,  0.1052, -1.1287],
         [ 0.0375, -0.2524,  0.0072],
         [ 0.0706, -0.1895, -0.2255],
         [ 0.0524, -0.0887, -0.2107],
         [ 0.0113, -0.1613,  0.0734]],

        [[ 0.1327, -0.9623,  0.0834],
         [ 0.1698, -0.6300, -0.3961],
         [ 0.1710, -0.2214, -0.7442],
         [ 0.1841, -0.149

Multi-Head Attention

In the previous section, we implemented causal (single-head) attention. Multi-head attention is formed by running multiple independent causal attention heads in parallel, not by stacking them sequentially. Each head has its own query, key, and value weight matrices (W_Q, W_K, and W_V).

A single attention head computes one attention distribution for each token, so it tends to focus on one dominant type of relationship at a time. Although it can learn useful patterns, its representational capacity is limited because all relationships must be expressed through the same attention mechanism.

By using multiple heads, each with its own learnable parameters, the model can learn different kinds of relationships simultaneously. For example, one head may focus on syntactic dependencies, another on semantic similarity, while another captures long-range context. The outputs of all heads are concatenated and projected to produce the final representation.

An intuitive analogy is neurons in a neural network used for image recognition. Different neurons often become sensitive to different features, such as edges, textures, or shapes. Similarly, different attention heads tend to specialize in different patterns or relationships in the input. While this analogy is not exact, it provides a useful intuition for why multiple attention heads are more expressive than a single head.

Efficient Multi-Head Attention

We can implement multi-head attention by creating multiple instances of the causal (single-head) attention module and running them in parallel. Each head has its own query, key, and value projection matrices and independently computes its attention output. Finally, the outputs of all heads are concatenated to produce the final representation.

Although conceptually simple, this approach is inefficient because every head performs separate linear transformations (W_Q, W_K, and W_V), resulting in multiple matrix multiplications.

The MultiHeadAttention class takes a more efficient approach. Instead of creating separate attention modules, it uses a single set of large projection matrices to compute the queries, keys, and values for all heads at once. This requires only one matrix multiplication for queries, one for keys, and one for values.

Initially, after the linear projection, the query, key, and value tensors have shape:

(batch_size, num_tokens, d_out)

where d_out = num_heads × head_dim.

To separate the individual heads, the tensor is reshaped using .view() into:

(batch_size, num_tokens, num_heads, head_dim)

This operation does not copy the data; it only changes how the tensor is viewed in memory.

Next, the tensor is transposed using .transpose(1, 2) to obtain:

(batch_size, num_heads, num_tokens, head_dim)

This layout places the num_heads dimension before the sequence length, allowing PyTorch to perform the attention computation for all heads simultaneously using efficient batched matrix multiplication.

Each head now independently computes:

Attention(Q, K, V)

Finally, the outputs from all heads are transposed back, reshaped to merge the head dimension into d_out, and passed through a final output projection (W_O) to produce the multi-head attention output.

Use of output projection:

Though output projection is optional most of the LLM use it. why because it create context with accounts to other headed output otherwise each headed output remain isolated and feed to LLM. for example Wo @ context = 0.1 * ah1 + 0.2 * ah2 ....

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, kqv_bias=True):
        super().__init__()
        assert d_out % num_heads == 0

        self.num_heads = num_heads
        self.heads_dim = d_out // num_heads
        self.d_out = d_out

        self.W_queries = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_values = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_keys = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))


    """ 
    Dim calculation

    input = (batch, ctx_len, emb_dim) = (5, 5, 3)
    d_in == emb_dim

    let d_out = 12 and num_heads = 4
    then heads_dim = 12 // 4 = 3

    for queries, values and keys
    queries = input @ W_queries = (5, 5, 3) @ (3, 12) = (5, 5, 12)
    values = input @ W_values = (5, 5, 3) @ (3, 12) = (5, 5, 12)
    keys = input @ W_keys = (5, 5, 3) @ (3, 12) = (5, 5, 12)

    12 is d_out(combined dim of head). but for further calc we need head as seperate dim.
    So, split the combined output dimension (d_out) into
    (num_heads, head_dim) using view().
    No data is copied; only the tensor shape changes.
    queries = (5, 5, 12) => (batch, ctx_len, num_heads, heads_dim) = (5, 5, 4, 3)
    values = (5, 5, 12) => (batch, ctx_len, num_heads, heads_dim) = (5, 5, 4, 3)
    keys = (5, 5, 12) => (batch, ctx_len, num_heads, heads_dim) = (5, 5, 4, 3)


    transpose so that we have num_head before ctx_len
    queries = (batch, ctx_len, num_heads, heads_dim) = (5, 5, 4, 3) => (batch, num_heads, ctx_len, heads_dim) = (5, 4, 5, 3)
    values = (batch, ctx_len, num_heads, heads_dim) = (5, 5, 4, 3) => (batch, num_heads, ctx_len, heads_dim) = (5, 4, 5, 3)
    keys = (batch, ctx_len, num_heads, heads_dim) = (5, 5, 4, 3) => (batch, num_heads, ctx_len, heads_dim) = (5, 4, 5, 3)

    attn_scores = queries @ keys.transpose(2, 3) = (5, 4, 5, 5), The first two dimensions (batch and head) act as batch dimensions.PyTorch performs one matrix multiplication for each (batch, head) pair independently.

    now, apply mask and scale by sqrt(dk)
    attn_weights = softmax(mask_fill(attn_scores) / sqrt(dk)) Dim(5, 4, 5, 5)
    dropout(attn_scores) (5, 4, 5, 5)

    context = attn_weights @ values => (5, 4, 5, 5) @ (5, 4, 5, 3) => (5, 4, 5, 3)
    context = context.transpose(1, 2) => (5, 5, 4, 3)
    context = context.view(5, 5, 12) => (batch, ctx_len, d_out)

    context = context @ out_proj => (5, 5, 12) @ (12, 12) => (5, 5, 12)
    """
    def forward(self, x):
        batch_dim, context_length, emb_dim = x.shape

        queries = self.W_queries(x)
        values = self.W_values(x)
        keys = self.W_keys(x)

        print("Queries         :", queries.shape)
        print("Keys            :", keys.shape)
        print("Values          :", values.shape)

        queries = queries.view(batch_dim, context_length, self.num_heads, self.heads_dim)
        values = values.view(batch_dim, context_length, self.num_heads, self.heads_dim)
        keys = keys.view(batch_dim, context_length, self.num_heads, self.heads_dim)

        print("After view      :", queries.shape)

        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        keys = keys.transpose(1, 2)
        print("After transpose :", queries.shape)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:context_length, :context_length]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        print("Attn scores     :", attn_scores.shape)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values)
        print("Context         :", context_vec.shape)
        context_vec = context_vec.transpose(1, 2)
        print("Transpose back  :", context_vec.shape)
        context_vec = context_vec.contiguous().view(batch_dim, context_length, self.d_out)
        print("After view      :", context_vec.shape)

        context_vec = self.out_proj(context_vec) 
        return context_vec




In [20]:
import torch

# Reproducibility
torch.manual_seed(123)

# Input
batch_size = 5
context_length = 5
d_in = 3
d_out = 12
num_heads = 4

x = torch.randn(batch_size, context_length, d_in)

# Model
mha = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=0.0,
    num_heads=num_heads,
    kqv_bias=False
)

# Forward pass
output = mha(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

assert output.shape == (batch_size, context_length, d_out)

print("\nOutput:")
print(output)

Queries         : torch.Size([5, 5, 12])
Keys            : torch.Size([5, 5, 12])
Values          : torch.Size([5, 5, 12])
After view      : torch.Size([5, 5, 4, 3])
After transpose : torch.Size([5, 4, 5, 3])
Attn scores     : torch.Size([5, 4, 5, 5])
Context         : torch.Size([5, 4, 5, 3])
Transpose back  : torch.Size([5, 5, 4, 3])
After view      : torch.Size([5, 5, 12])
Input shape : torch.Size([5, 5, 3])
Output shape: torch.Size([5, 5, 12])

Output:
tensor([[[-0.2373, -0.3189,  0.0766, -0.1331,  0.1865,  0.2342,  0.1632,
           0.3449, -0.0767,  0.2600, -0.1263, -0.0151],
         [-0.2534, -0.2640,  0.1491, -0.1031,  0.1675,  0.2543,  0.0684,
           0.1781, -0.0613,  0.2303, -0.1442, -0.0678],
         [-0.1836, -0.2913,  0.1916, -0.0946,  0.0359,  0.2596, -0.0550,
           0.1290, -0.0114,  0.2263, -0.1503, -0.0177],
         [-0.3622, -0.2830,  0.0031, -0.1550,  0.3801,  0.1963,  0.1730,
           0.4117, -0.1084,  0.3925, -0.2008,  0.0102],
         [-0.3629, -0.1